# Clusterização de texto: descrições de itens de nota fiscal

Roteiro de estudo com a base sintética `base_cluster_estudo.xlsx`.
Rode célula por célula e mexa nos parâmetros marcados com `# <<< TESTE`.

**Regra do jogo:** até a seção 7, finja que o gabarito não existe. Em um projeto real
você não tem a "resposta certa", então precisa julgar os clusters olhando os termos,
exemplos e métricas internas (silhouette etc.).

Bibliotecas: pandas, numpy, scikit-learn, scipy, matplotlib, seaborn, openpyxl.

In [0]:
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import (DBSCAN, HDBSCAN, AgglomerativeClustering, KMeans,
                             MiniBatchKMeans)
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import (adjusted_rand_score, davies_bouldin_score,
                             homogeneity_completeness_v_measure,
                             normalized_mutual_info_score, silhouette_samples,
                             silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200, "display.max_colwidth", 70)

ARQUIVO = "base_cluster_estudo.xlsx"
GABARITO = "gabarito_cluster_estudo.csv"   # só para avaliar (seção 7 em diante)
SEED = 42

## 1. Carregar os dados

In [0]:
df = pd.read_excel(ARQUIVO)
gab = pd.read_csv(GABARITO)
assert len(df) == len(gab) and (df["DESC_ITEM_NF"] == gab["DESC_ITEM_NF"]).all()
y_true = gab["categoria"].values        # NÃO usar nas seções 2 a 6
y_sub = gab["subcategoria"].values
print(df.shape)
df.sample(10, random_state=1)

## 2. Exploração rápida
O que muda a escolha de técnica: tamanho dos textos, vocabulário, quantidade de códigos.

In [0]:
df["n_chars"] = df["DESC_ITEM_NF"].str.len()
df["n_tokens"] = df["DESC_ITEM_NF"].str.split().str.len()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(df["n_chars"], bins=30, ax=ax[0]).set(title="Caracteres por descrição")
sns.histplot(df["n_tokens"], bins=range(1, 14), ax=ax[1]).set(title="Palavras por descrição")
plt.tight_layout()
plt.show()

tokens = df["DESC_ITEM_NF"].str.split().explode()
print("vocabulário:", tokens.nunique(), "| tokens que aparecem 1 única vez:", (tokens.value_counts() == 1).sum())
print("% de tokens só numéricos:", round(tokens.str.fullmatch(r"\d+").mean() * 100, 1))
print("% de tokens com letra+número (ex.: s10, 15w40, cod123):",
      round(tokens.str.contains(r"(?=.*\d)(?=.*[a-z])").mean() * 100, 1))
tokens.value_counts().head(25)

## 3. Pré-processamento
Três decisões que mudam o resultado:
1. **Medidas** (`200ml`, `1l`, `500g`): virar um token por tipo de unidade?
2. **Códigos** (`9470`, `ref5959`): jogar fora? Cuidado: `1202` (ONU do diesel) é informativo.
3. **Abreviações** (`choc`, `deterg`): expandir com dicionário?

In [0]:
MEDIDA = re.compile(r"\b\d+\s?(ml|lts|lt|l|litros?|kg|gr|g|un|mm|cm|mts|m|w|v|ah|k|pol)\b")
NORMALIZA_UNIDADE = {"lts": "l", "lt": "l", "litro": "l", "litros": "l", "gr": "g"}
MANTER_NUMEROS = {"1202", "1203", "1170"}          # códigos ONU de combustíveis
EXPANDE = {"choc": "chocolate", "refri": "refrigerante", "refrig": "refrigerante",
           "desinf": "desinfetante", "deterg": "detergente", "sab": "sabonete",
           "bisc": "biscoito", "adit": "aditivo", "aditv": "aditivo", "aditiv": "aditivado",
           "comb": "combustivel", "combust": "combustivel", "hid": "hidratado",
           "hidrat": "hidratado", "etil": "etilico", "lamp": "lampada", "filt": "filtro",
           "elem": "elemento", "diant": "dianteiro", "tras": "traseiro", "lubrif": "lubrificante",
           "sint": "sintetico", "liq": "liquido", "verm": "vermelho", "bco": "branco"}


def limpar(s, medidas=True, codigos=True, expandir=True):
    if medidas:
        s = MEDIDA.sub(lambda m: f" med_{NORMALIZA_UNIDADE.get(m.group(1), m.group(1))} ", s)
    toks = s.split()
    if expandir:
        toks = [EXPANDE.get(t, t) for t in toks]
    if codigos:
        toks = [t for t in toks
                if not (re.fullmatch(r"\d{3,}", t) and t not in MANTER_NUMEROS)
                and not re.fullmatch(r"(cod|ref|sku)\d+", t)]
    return " ".join(toks)


df["texto"] = df["DESC_ITEM_NF"].apply(limpar)
df[["DESC_ITEM_NF", "texto"]].sample(12, random_state=3)

## 4. Vetorização: palavras vs. caracteres
Em texto sujo (typos, palavras coladas, truncadas) o TF-IDF de **n-gramas de caracteres**
*tende* a ser mais robusto que o de palavras: `diesil`, `desel` e `diesel` compartilham
vários trigramas, mas são três "palavras" diferentes. Veja na tabela se isso se confirma aqui
e se o pré-processamento (`texto` limpo vs. `bruto`) fez diferença.
(Cada combinação passa por um SVD de 30 dimensões antes do KMeans; a razão está na próxima seção.)

In [0]:
def vetorizar(textos, tipo):
    if tipo == "palavra":
        v = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    else:  # char
        v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)
    return v.fit_transform(textos), v


K_REF = 15   # <<< TESTE (aqui usamos o nº de categorias do gerador; na vida real você não sabe)
linhas = []
for prep, col in [("bruto", "DESC_ITEM_NF"), ("limpo", "texto")]:
    for tipo in ["palavra", "char"]:
        X_, _ = vetorizar(df[col], tipo)
        X_ = normalize(TruncatedSVD(30, random_state=SEED).fit_transform(X_))   # ver seção 4b
        lab = KMeans(K_REF, n_init=5, random_state=SEED).fit_predict(X_)
        linhas.append({"texto": prep, "vetor": tipo,
                       "ARI": adjusted_rand_score(y_true, lab),
                       "NMI": normalized_mutual_info_score(y_true, lab)})
pd.DataFrame(linhas).round(3)

> Aqui o gabarito foi usado só para comparar combinações de pré-processamento.
> Isso é "trapaça didática": em produção você compararia com amostra rotulada na mão
> ou com métricas internas.

In [0]:
X, vec = vetorizar(df["texto"], "char")      # <<< TESTE: troque por "palavra"
print("matriz esparsa (documentos x n-gramas):", X.shape)

### Reduzir dimensão (SVD) antes de clusterizar
O TF-IDF tem milhares de colunas quase todas zero. O `TruncatedSVD` comprime isso em poucas
dimensões densas (é o "LSA"). **Quantas dimensões?** Isso muda muito o resultado, então teste:

> Atenção: o silhouette **não é comparável** entre dimensões diferentes (com menos dimensões ele
> sobe quase sempre). Use-o para comparar valores de `k` na *mesma* representação.

In [0]:
linhas = []
for d in [10, 20, 30, 50, 100, 200]:
    Zd = normalize(TruncatedSVD(d, random_state=SEED).fit_transform(X))
    lab = KMeans(K_REF, n_init=10, random_state=SEED).fit_predict(Zd)
    linhas.append({"dimensoes": d, "ARI": adjusted_rand_score(y_true, lab),
                   "NMI": normalized_mutual_info_score(y_true, lab),
                   "silhouette (interno)": silhouette_score(Zd, lab, metric="cosine")})
pd.DataFrame(linhas).round(3)

In [0]:
N_DIM = 30   # <<< TESTE
svd = TruncatedSVD(N_DIM, random_state=SEED)
Z = normalize(svd.fit_transform(X))            # vetores densos de norma 1
print("reduzida:", Z.shape, "| variância explicada:", round(svd.explained_variance_ratio_.sum(), 2))

## 5. Quantos clusters? (sem gabarito)
Três olhares: cotovelo (inércia), silhouette (maior é melhor), Davies-Bouldin (menor é melhor).
Nenhum dá a resposta sozinho. Em texto é comum **não haver um cotovelo claro** (aqui o silhouette
tende a subir com `k`): use como pista e decida também por interpretabilidade, olhando os exemplos.